# 05 — Honest Model Comparison & Production Readiness Gate
# So sánh Mô hình Trung thực & Cổng Đánh giá Sẵn sàng Sản xuất

**English:**
Loads saved JSON/CSV evaluation artifacts strictly from Google Drive (`/content/drive/MyDrive/mimamori_companion_ai/eval_outputs`) and local `ai-training/outputs/` to compare:
1. **Base `Qwen/Qwen3-4B-Instruct-2507`**
2. **LoRA-tuned `Qwen/Qwen3-4B-Instruct-2507`**
3. **`Qwen/Qwen3.5-4B`**
4. **Groq Fallback (`openai/gpt-oss-120b` -> `openai/gpt-oss-20b`)**

**Tiếng Việt:**
Tổng hợp và so sánh kết quả đánh giá **hoàn toàn dựa trên các file JSON/CSV đã lưu** trên Google Drive (hoặc thư mục `outputs/`). Mọi mô hình chưa chạy hoặc gặp lỗi được hiển thị trung thực là `NOT_TESTED` / `NOT_SUPPORTED` / `FAILED` — **tuyệt đối không tự bịa đặt hay nội suy điểm số**.

> **Zero Fabrication Policy / Quy tắc Chống Bịa đặt Điểm:**
> - Only models with `execution_status == "TESTED_SUCCESSFULLY"` and `is_mock == False` enter real model comparisons and charts.
> - `mock_dev` (`is_mock: true`) is shown separately strictly as a pipeline-verification reference.


## Step 1: Mount Drive & Discover Saved Evaluation Artifacts
## Bước 1: Kết nối Drive & Tìm các File Kết quả Đánh giá Đã lưu


In [ ]:
import os
import sys
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT_DEFAULT = Path("/content/drive/MyDrive/mimamori_companion_ai")

# Optional: set AI_TRAINING_DIR explicitly if your repo lives in a custom folder
CUSTOM_AI_TRAINING_DIR = os.environ.get("MIMAMORI_AI_TRAINING_DIR", "").strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("[Drive] Google Drive mounted at /content/drive")
else:
    print("[Local] Not running inside Google Colab; Drive mount skipped.")


def resolve_ai_training_dir() -> Path:
    candidates: list[Path] = []
    if CUSTOM_AI_TRAINING_DIR:
        candidates.append(Path(CUSTOM_AI_TRAINING_DIR))
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / "ai-training",
        cwd.parent / "ai-training",
        Path("/content/mimamori/ai-training"),
        Path("/content/ai-training"),
        Path("/content/drive/MyDrive/mimamori/ai-training"),
        Path("/content/drive/MyDrive/mimamori_companion_ai/repo/ai-training"),
        Path("/content/drive/MyDrive/mimamori_companion_ai/ai-training"),
    ])
    for cand in candidates:
        if (cand / "configs" / "model_config.yaml").exists() and (cand / "configs" / "training_config.yaml").exists():
            return cand.resolve()
    raise FileNotFoundError(
        "Could not locate 'ai-training/configs/model_config.yaml'. "
        "In Colab, clone/upload the repository to /content/mimamori or Google Drive, "
        "or set CUSTOM_AI_TRAINING_DIR / MIMAMORI_AI_TRAINING_DIR."
    )


AI_TRAINING_DIR = resolve_ai_training_dir()
if str(AI_TRAINING_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(AI_TRAINING_DIR / "scripts"))

import yaml

MODEL_CONFIG_PATH = AI_TRAINING_DIR / "configs" / "model_config.yaml"
TRAINING_CONFIG_PATH = AI_TRAINING_DIR / "configs" / "training_config.yaml"
with MODEL_CONFIG_PATH.open("r", encoding="utf-8") as f:
    MODEL_CFG = yaml.safe_load(f)
with TRAINING_CONFIG_PATH.open("r", encoding="utf-8") as f:
    TRAIN_CFG = yaml.safe_load(f)

PATHS_CFG = TRAIN_CFG["paths"]
if IN_COLAB:
    DRIVE_ROOT = Path(PATHS_CFG["drive_root"])
    DRIVE_CHECKPOINTS_DIR = Path(PATHS_CFG["drive_checkpoints_dir"])
    DRIVE_FINAL_ADAPTER_DIR = Path(PATHS_CFG["drive_final_adapter_dir"])
    DRIVE_EVAL_OUTPUTS_DIR = Path(PATHS_CFG["drive_eval_outputs_dir"])
    DRIVE_PROCESSED_DATA_DIR = Path(PATHS_CFG["drive_processed_data_dir"])
else:
    local_drive_sim = AI_TRAINING_DIR / "outputs" / "_local_drive_mirror"
    DRIVE_ROOT = local_drive_sim
    DRIVE_CHECKPOINTS_DIR = local_drive_sim / "checkpoints"
    DRIVE_FINAL_ADAPTER_DIR = local_drive_sim / "adapters" / "qwen3-4b-companion-lora"
    DRIVE_EVAL_OUTPUTS_DIR = AI_TRAINING_DIR / PATHS_CFG["local_outputs_dir"]
    DRIVE_PROCESSED_DATA_DIR = AI_TRAINING_DIR / "datasets" / "processed"

for d in (DRIVE_ROOT, DRIVE_CHECKPOINTS_DIR, DRIVE_FINAL_ADAPTER_DIR, DRIVE_EVAL_OUTPUTS_DIR, DRIVE_PROCESSED_DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"[Repo] AI_TRAINING_DIR      = {AI_TRAINING_DIR}")
print(f"[Drive] DRIVE_ROOT          = {DRIVE_ROOT}")
print(f"[Drive] DRIVE_EVAL_OUTPUTS  = {DRIVE_EVAL_OUTPUTS_DIR}")
print(f"[Drive] DRIVE_PROCESSED     = {DRIVE_PROCESSED_DATA_DIR}")


## Step 2: Load & Consolidate Real Target Results (No Score Fabrication)
## Bước 2: Đọc & Hợp nhất Kết quả Thực tế (Không Bịa đặt Điểm)

Inspects `baseline_eval_results.json`, `baseline_qwen3_results.json`, `baseline_qwen35_results.json`, `baseline_groq_results.json`, `lora_qwen3_eval_results.json`, and `eval_results.json` (plus `training_metadata.json`).


In [ ]:
import csv
import json
from evaluate_model import METRIC_NAMES, empty_metrics

SEARCH_DIRS = [DRIVE_EVAL_OUTPUTS_DIR]
local_out = AI_TRAINING_DIR / PATHS_CFG["local_outputs_dir"]
if local_out.resolve() != DRIVE_EVAL_OUTPUTS_DIR.resolve():
    SEARCH_DIRS.append(local_out)

print("Searching evaluation outputs in:", [str(d) for d in SEARCH_DIRS])


def _load_json_if_exists(filename: str) -> dict | None:
    for d in SEARCH_DIRS:
        p = d / filename
        if p.exists():
            print(f"[Loaded] {p}")
            return json.loads(p.read_text(encoding="utf-8"))
    return None


def _pick_target_entry(payload: dict | None, target_name: str) -> dict | None:
    if not payload:
        return None
    for entry in payload.get("targets", []):
        if entry.get("target") == target_name:
            return entry
    return None


STATUS_PRIORITY = {
    "TESTED_SUCCESSFULLY": 4,
    "FAILED": 3,
    "NOT_SUPPORTED": 2,
    "NOT_TESTED": 1,
}


def _select_best_record(candidates: list[dict | None], default_label: str, default_model_id: str, is_mock: bool = False) -> dict:
    valid = [c for c in candidates if c is not None]
    if not valid:
        return {
            "label": default_label,
            "model_id": default_model_id,
            "is_mock": is_mock,
            "execution_status": "NOT_TESTED",
            "status_reason": "No evaluation JSON artifact found on Drive or local outputs/.",
            "cases_evaluated": 0,
            "summary_metrics": empty_metrics(),
            "case_results": [],
        }
    best = max(valid, key=lambda x: STATUS_PRIORITY.get(x.get("execution_status", "NOT_TESTED"), 0))
    status = best.get("execution_status", "NOT_TESTED")
    metrics = dict(best.get("summary_metrics") or empty_metrics())
    if status != "TESTED_SUCCESSFULLY":
        metrics = empty_metrics()
    return {
        "label": default_label,
        "model_id": best.get("model_id", default_model_id),
        "is_mock": bool(best.get("is_mock", is_mock)),
        "execution_status": status,
        "status_reason": best.get("status_reason", ""),
        "cases_evaluated": int(best.get("cases_evaluated", 0)),
        "summary_metrics": metrics,
        "case_results": list(best.get("case_results", [])),
    }


baseline_all = _load_json_if_exists("baseline_eval_results.json")
baseline_qwen3 = _load_json_if_exists("baseline_qwen3_results.json")
baseline_qwen35 = _load_json_if_exists("baseline_qwen35_results.json")
baseline_groq = _load_json_if_exists("baseline_groq_results.json")
lora_eval = _load_json_if_exists("lora_qwen3_eval_results.json")
generic_eval = _load_json_if_exists("eval_results.json")
training_meta = _load_json_if_exists("training_metadata.json")
test_split_eval = _load_json_if_exists("test_split_eval.json")

# Note: generic_eval may contain mock_dev only (or real targets if user ran evaluate_model.py directly without --output-prefix)
# Only treat generic_eval qwen3_4b_instruct_2507 as Base if "+adapter:" is not in model_id
gen_q3 = _pick_target_entry(generic_eval, "qwen3_4b_instruct_2507")
gen_q3_base = gen_q3 if (gen_q3 and "+adapter:" not in str(gen_q3.get("model_id", ""))) else None
gen_q3_lora = gen_q3 if (gen_q3 and "+adapter:" in str(gen_q3.get("model_id", ""))) else None

comparison_rows = [
    _select_best_record(
        [_pick_target_entry(baseline_qwen3, "qwen3_4b_instruct_2507"), _pick_target_entry(baseline_all, "qwen3_4b_instruct_2507"), gen_q3_base],
        "Base Qwen3-4B-Instruct-2507",
        MODEL_CFG["primary_model"]["id"],
    ),
    _select_best_record(
        [_pick_target_entry(lora_eval, "qwen3_4b_instruct_2507"), gen_q3_lora],
        "LoRA Qwen3-4B-Instruct-2507",
        f"{MODEL_CFG['primary_model']['id']} (+adapter)",
    ),
    _select_best_record(
        [_pick_target_entry(baseline_qwen35, "qwen3_5_4b"), _pick_target_entry(baseline_all, "qwen3_5_4b"), _pick_target_entry(generic_eval, "qwen3_5_4b")],
        "Qwen/Qwen3.5-4B",
        MODEL_CFG["comparison_model"]["id"],
    ),
    _select_best_record(
        [_pick_target_entry(baseline_groq, "groq_fallback"), _pick_target_entry(baseline_all, "groq_fallback"), _pick_target_entry(generic_eval, "groq_fallback")],
        "Groq Fallback",
        f"{MODEL_CFG['fallback_llm']['primary_model']} -> {MODEL_CFG['fallback_llm']['fallback_model']}",
    ),
]

mock_record = _select_best_record(
    [_pick_target_entry(generic_eval, "mock_dev")],
    "mock_dev (pipeline smoke only)",
    "local-deterministic-mock",
    is_mock=True,
)


## Step 3: Display Comparison Matrix, Base-vs-LoRA Deltas & Honest Chart
## Bước 3: Bảng So sánh, Mức Cải thiện Base vs LoRA & Biểu đồ Trung thực


In [ ]:
def _fmt_metric(val) -> str:
    if val is None:
        return "NOT_TESTED"
    return f"{val:.4f}" if isinstance(val, float) and val <= 10.0 else f"{val:.2f}"


print("=== Real Model Comparison Table ===")
header = f"{'Model':30s} | {'Status':19s} | {'Cases':5s} | " + " | ".join(f"{m[:10]:>10s}" for m in METRIC_NAMES)
print(header)
print("-" * len(header))
for row in comparison_rows:
    m_str = " | ".join(
        f"{('—' if row['summary_metrics'].get(m) is None else _fmt_metric(row['summary_metrics'].get(m))):>10s}"
        for m in METRIC_NAMES
    )
    print(f"{row['label']:30s} | {row['execution_status']:19s} | {row['cases_evaluated']:5d} | {m_str}")

print("\n--- Status Reasons ---")
for row in comparison_rows:
    print(f"  * {row['label']}: [{row['execution_status']}] {row['status_reason']}")

if mock_record["execution_status"] == "TESTED_SUCCESSFULLY":
    print(f"\n[Pipeline Verification Only — NOT a Real LLM] mock_dev ({mock_record['cases_evaluated']} cases): {mock_record['summary_metrics']}")

# Base vs LoRA Delta (only if BOTH are TESTED_SUCCESSFULLY)
base_row, lora_row = comparison_rows[0], comparison_rows[1]
if base_row["execution_status"] == "TESTED_SUCCESSFULLY" and lora_row["execution_status"] == "TESTED_SUCCESSFULLY":
    print("\n=== Base vs LoRA-Tuned Qwen3-4B Delta (LoRA - Base) ===")
    for m in METRIC_NAMES:
        b_val = base_row["summary_metrics"].get(m)
        l_val = lora_row["summary_metrics"].get(m)
        if b_val is not None and l_val is not None:
            diff = l_val - b_val
            print(f"  {m:24s}: base={b_val:.4f} -> lora={l_val:.4f} (delta={diff:+.4f})")
else:
    print(
        f"\n[Delta Unavailable] Base status={base_row['execution_status']}, LoRA status={lora_row['execution_status']}. "
        "Run notebooks 03 and 04 on Colab T4 to obtain real Base vs LoRA deltas."
    )

# Optional chart strictly for real tested models
tested_real = [r for r in comparison_rows if r["execution_status"] == "TESTED_SUCCESSFULLY" and not r["is_mock"]]
if tested_real:
    try:
        import matplotlib.pyplot as plt
        plot_metrics = [
            "vietnamese_naturalness",
            "empathy",
            "clarity_for_elderly",
            "instruction_following",
            "medical_safety",
            "medication_accuracy",
        ]
        x = list(range(len(plot_metrics)))
        width = 0.8 / max(1, len(tested_real))
        plt.figure(figsize=(11, 4.5))
        for idx, r in enumerate(tested_real):
            vals = [float(r["summary_metrics"].get(m) or 0.0) for m in plot_metrics]
            offsets = [xi + idx * width for xi in x]
            plt.bar(offsets, vals, width=width, label=r["label"])
        plt.xticks([xi + width * (len(tested_real) - 1) / 2 for xi in x], plot_metrics, rotation=20, ha="right")
        plt.ylim(0.0, 1.1)
        plt.ylabel("Score (0.0 - 1.0)")
        plt.title("Mimamori Companion AI — Real Tested Models Comparison")
        plt.legend()
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(DRIVE_EVAL_OUTPUTS_DIR / "model_comparison_chart.png", dpi=150)
        plt.show()
    except Exception as exc:
        print(f"[Info] Chart skipped: {exc}")
else:
    print("\n[Chart Skipped] No real GPU/API models have TESTED_SUCCESSFULLY outputs yet (refusing to plot fabricated or mock scores).")


## Step 4: Qualitative Side-by-Side Comparison & Production Acceptance Gate
## Bước 4: So sánh Phản hồi Thực tế & Cổng Duyệt Đưa vào Sản xuất

Checks the production acceptance rules from `docs/companion-ai-evaluation.md`:
1. Clinical/geriatric review completed (`reviewed=true`, recommended `>= 200` training examples)
2. `medical_safety == 1.0` and `hallucination == 0.0` on critical safety prompts
3. `0` missed `115` + caregiver escalations on emergencies


In [ ]:
# Side-by-side qualitative inspection for tested real models (selected dynamically by category)
tested_with_cases = [r for r in comparison_rows if r["execution_status"] == "TESTED_SUCCESSFULLY" and r["case_results"]]
if tested_with_cases:
    ref_cases = tested_with_cases[0]["case_results"]
    sample_ids = []
    for target_cat in ("emergency_situations", "medication_assistance", "basic_health_information", "unclear_or_insufficient_information"):
        found = next((c["id"] for c in ref_cases if c.get("category") == target_cat and c["id"] not in sample_ids), None)
        if found:
            sample_ids.append(found)
    for cid in sample_ids:
        print(f"\n--- Benchmark Case {cid} ---")
        for r in tested_with_cases:
            match = next((c for c in r["case_results"] if c["id"] == cid), None)
            if match:
                print(f"  [{r['label']}] cat={match['category']} | safety={match['metrics']['medical_safety']} | {match['response']}")
else:
    print("[Qualitative Inspection] No real model case_results available yet.")

# Write honest comparison summary CSV + Markdown to DRIVE_EVAL_OUTPUTS_DIR
summary_csv = DRIVE_EVAL_OUTPUTS_DIR / "model_comparison_summary.csv"
fieldnames = ["label", "model_id", "is_mock", "execution_status", "cases_evaluated", *METRIC_NAMES, "status_reason"]
with summary_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in comparison_rows:
        row_dict = {
            "label": r["label"],
            "model_id": r["model_id"],
            "is_mock": r["is_mock"],
            "execution_status": r["execution_status"],
            "cases_evaluated": r["cases_evaluated"],
            "status_reason": r["status_reason"],
        }
        for m in METRIC_NAMES:
            v = r["summary_metrics"].get(m) if r["execution_status"] == "TESTED_SUCCESSFULLY" else None
            row_dict[m] = "" if v is None else v
        writer.writerow(row_dict)
print(f"\n[Saved Comparison CSV] {summary_csv}")

md_lines = [
    "# Mimamori Companion AI — Model Comparison Report",
    "",
    "| Model | Status | Cases | " + " | ".join(METRIC_NAMES) + " |",
    "|---|---|---:|" + "|".join("---:" for _ in METRIC_NAMES) + "|",
]
for r in comparison_rows:
    vals = [
        "—" if (r["execution_status"] != "TESTED_SUCCESSFULLY" or r["summary_metrics"].get(m) is None)
        else _fmt_metric(r["summary_metrics"].get(m))
        for m in METRIC_NAMES
    ]
    md_lines.append(f"| {r['label']} | `{r['execution_status']}` | {r['cases_evaluated']} | " + " | ".join(vals) + " |")
summary_md = DRIVE_EVAL_OUTPUTS_DIR / "model_comparison_report.md"
summary_md.write_text("\n".join(md_lines) + "\n", encoding="utf-8")
print(f"[Saved Comparison MD ] {summary_md}")

if training_meta:
    prov = training_meta.get("dataset_provenance", {})
    print("\n--- Training Provenance & Promotion Gate ---")
    print("  Production approved?   :", prov.get("production_approved", False))
    print("  Is synthetic smoke?    :", prov.get("is_synthetic_smoke_test", True))
    print("  Reviewed train count   :", prov.get("reviewed_train_count", 0))
    print("  Effective T4 profile   :", training_meta.get("effective_profile"))
if test_split_eval:
    print("  Test split eval metrics:", test_split_eval.get("metrics"))
